# 11 — DeepFM Fine-Ranking (MVP First)

PyTorch DeepFM on Notebook 09's Top-500 candidate pool, compared with Notebook 10 RRF / Binary / LambdaRank. Validation-only early stopping, score fusion and feature ablation. **Test is evaluated once after configuration is frozen.**

**Environment:** Windows Conda `torch` kernel. Install missing dependencies with `%pip install torch pandas pyarrow scikit-learn duckdb matplotlib`.

**Caveat:** Candidate non-positives are *weak negatives*, not observed non-click impressions. Results are offline ranking metrics, not CTR/CVR estimates. Split-level PIT, as in Notebook 09.

In [1]:
from pathlib import Path
import json, time, random, copy, gc
import numpy as np, pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score
from IPython.display import display
import duckdb

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Avoid Windows DataLoader multiprocessing pitfalls.
NUM_WORKERS=0
cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,*cwd.parents) if (p/'data'/'processed'/'notebook02.duckdb').exists()),None)
assert ROOT is not None,'Open the project root or notebooks/ directory; notebook02.duckdb not found.'
OUT=ROOT/'data'/'processed'; REPORT=ROOT/'data'/'reports'; MODEL_DIR=ROOT/'models'
REPORT.mkdir(parents=True,exist_ok=True);MODEL_DIR.mkdir(parents=True,exist_ok=True)
SPLITS=('train','validation','test'); KS=(20,50,100)
manifest=json.loads((REPORT/'notebook09_feature_manifest.json').read_text(encoding='utf-8'))
FEATURES=list(manifest['feature_columns']); CATS=list(manifest['categorical_columns'])
NUMS=[x for x in FEATURES if x not in CATS]
assert len(FEATURES)==57 and len(CATS)==3
paths={s:OUT/f'{s}_ranking_features_top500.parquet' for s in SPLITS}
for p in paths.values():assert p.exists(),f'Missing {p}'
print('ROOT:',ROOT,'DEVICE:',DEVICE,'Features:',len(FEATURES),'Numerical:',len(NUMS),'Categorical:',CATS)

ROOT: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks DEVICE: cuda Features: 57 Numerical: 54 Categorical: ['category_id', 'parent_category_id', 'root_category_id']


## 1. Load features and baseline predictions

Read only required columns; sort by user and original retrieval rank. IDs and labels are **not** model features. Reuse Notebook 10 predictions for exact same candidate pool; assert alignment by `(user_id,item_id)`.

In [2]:
BASE_METHODS={'rrf_baseline':'rrf_baseline','binary':'binary','lambdarank':'lambdarank'}
DATA={}; BASE={}; audit=[]
for s in SPLITS:
    cols=list(dict.fromkeys(['user_id','item_id','label','retrieval_rank']+FEATURES))
    d=pd.read_parquet(paths[s],columns=cols)
    d=d.sort_values(['user_id','retrieval_rank'],kind='stable').reset_index(drop=True)
    assert not d[['user_id','item_id']].duplicated().any()
    assert d.label.isin([0,1]).all()
    counts=d.groupby('user_id',sort=False).size()
    assert counts.min()==counts.max()==500
    assert d.retrieval_rank.between(1,500).all()
    DATA[s]=d
    BASE[s]={}
    for method,suffix in BASE_METHODS.items():
        p=OUT/f'{s}_lightgbm_{suffix}_top500.parquet'
        assert p.exists(),f'Missing {p}; run Notebook 10.'
        pred=pd.read_parquet(p)
        print(s,method,'prediction columns:',list(pred.columns)) if s=='train' else None
        assert {'user_id','item_id','score'}<=set(pred.columns),f'{p}: missing columns'
        pred=pred[['user_id','item_id','score']].rename(columns={'score':'baseline_score'})
        assert not pred[['user_id','item_id']].duplicated().any()
        aligned=d[['user_id','item_id']].merge(pred,on=['user_id','item_id'],how='left',validate='one_to_one',sort=False)
        assert len(aligned)==len(d) and aligned.baseline_score.notna().all(),f'{s}/{method}: misaligned candidates'
        BASE[s][method]=aligned.baseline_score.to_numpy(dtype=np.float32)
    audit.append(dict(split=s,rows=len(d),users=len(counts),positives=int(d.label.sum()),positive_rate=float(d.label.mean()),group_min=int(counts.min()),group_max=int(counts.max())))
display(pd.DataFrame(audit))

train rrf_baseline prediction columns: ['user_id', 'item_id', 'label', 'score', 'original_rank', 'model_rank']
train binary prediction columns: ['user_id', 'item_id', 'label', 'score', 'original_rank', 'model_rank']
train lambdarank prediction columns: ['user_id', 'item_id', 'label', 'score', 'original_rank', 'model_rank']


,split,rows,users,positives,positive_rate,group_min,group_max
0,train,2055000,4110,2131,0.001037,500,500
1,validation,1828500,3657,1996,0.001092,500,500
2,test,1786000,3572,1785,0.000999,500,500


## 2. Preprocessing: train-only numeric statistics and category vocabulary

Numeric missing values are imputed using **train medians**, log1p is applied only to train-observed nonnegative heavy-tailed features, then values are clipped/standardized using **train-only** statistics. Category IDs are mapped using **train-only** vocabularies; unseen values use OOV=0. No validation/test distribution is fitted.

In [3]:
# Free the source DataFrames only after preprocessing/evaluation is complete.
# Numeric transform fitted exclusively on train.
train_num=DATA['train'][NUMS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
med=train_num.median().fillna(0).to_numpy(dtype=np.float64)
tr=train_num.to_numpy(dtype=np.float64,copy=True)
tr=np.where(np.isfinite(tr),tr,med)
# Log-transform only nonnegative columns with large positive tails (train data only).
logmask=np.array([(tr[:,j].min()>=0 and np.nanpercentile(tr[:,j],99)>20) for j in range(tr.shape[1])])
tr[:,logmask]=np.log1p(np.maximum(tr[:,logmask],0))
lo=np.quantile(tr,.001,axis=0);hi=np.quantile(tr,.999,axis=0)
tr=np.clip(tr,lo,hi)
mu=tr.mean(axis=0);sigma=np.maximum(tr.std(axis=0),1e-5)
del train_num,tr
# Train-only category vocabularies; missing/unseen category => OOV 0.
VOCABS={}
for c in CATS:
    values=pd.to_numeric(DATA['train'][c],errors='coerce').fillna(-1).astype('int64').unique()
    valid_values=sorted(int(v) for v in values if int(v)>=0)
    VOCABS[c]={v:i+1 for i,v in enumerate(valid_values)}
CARD=[len(VOCABS[c])+1 for c in CATS]

def transform(s):
    df=DATA[s]
    x=df[NUMS].to_numpy(dtype=np.float64,copy=True)
    x=np.where(np.isfinite(x),x,med)
    x[:,logmask]=np.log1p(np.maximum(x[:,logmask],0))
    x=np.clip(x,lo,hi)
    x=((x-mu)/sigma).astype('float32')
    x=np.clip(x,-8,8)
    cat=np.stack([pd.to_numeric(df[c],errors='coerce').fillna(-1).astype('int64').map(VOCABS[c]).fillna(0).to_numpy(dtype=np.int64) for c in CATS],axis=1)
    assert np.isfinite(x).all() and (cat>=0).all()
    return x,cat
XNUM={};XCAT={};Y={}
for s in SPLITS:
    XNUM[s],XCAT[s]=transform(s)
    Y[s]=DATA[s].label.to_numpy(dtype=np.float32)
for s in SPLITS:
    for j,c in enumerate(CATS):
        col=XCAT[s][:,j]
        assert col.min()>=0 and col.max()<CARD[j], (s,c,int(col.min()),int(col.max()),CARD[j])
print('Embedding index range validation: PASS')
print('Numerical fields:',len(NUMS),'Category cardinalities:',CARD,'Log-transformed:',[NUMS[i] for i,v in enumerate(logmask) if v])

Embedding index range validation: PASS
Numerical fields: 54 Category cardinalities: [904, 256, 21] Log-transformed: ['retrieval_rank', 'popularity_rank', 'popularity_score', 'cf_rank', 'faiss_rank', 'category_rank', 'category_score', 'user_events_30d', 'user_unique_items_30d', 'user_events_7d', 'user_views_30d', 'user_views_7d', 'user_recency_days', 'item_events_30d', 'item_users_30d', 'item_events_7d', 'item_events_1d', 'item_views_30d', 'item_carts_30d', 'item_views_7d', 'item_recency_days', 'known_property_count', 'days_since_property_update', 'property_distinct_keys_historical', 'category_30d_events']


## 3. Evaluation: all eligible users and positives

Recall is divided by Notebook 02's eligible future positives, not by candidate positives. NDCG uses the same full-eligible denominator as Notebook 10. PR-AUC is average precision on the fixed 500/user candidate pool.

In [4]:
con=duckdb.connect(str(OUT/'notebook02.duckdb'),read_only=True)
POS={};ELIGIBLE={}
for s in SPLITS:
    counts=con.execute(f'SELECT user_id, COUNT(*) n FROM {s}_positives GROUP BY user_id').df()
    POS[s]=dict(zip(counts.user_id.astype('int64'),counts.n.astype('int32')))
    ELIGIBLE[s]=int(counts.n.sum())
    assert len(POS[s])==DATA[s].user_id.nunique()

def evaluate(s,method,scores):
    d=DATA[s];scores=np.asarray(scores,dtype=np.float64)
    assert len(scores)==len(d) and np.isfinite(scores).all()
    order=pd.DataFrame({'user_id':d.user_id.to_numpy(),'item_id':d.item_id.to_numpy(),
        'label':Y[s].astype(np.int8),'score':scores,'original_rank':d.retrieval_rank.to_numpy()})
    order=order.sort_values(['user_id','score','original_rank','item_id'],ascending=[True,False,True,True],kind='stable').reset_index(drop=True)
    order['model_rank']=order.groupby('user_id',sort=False).cumcount()+1
    u=order.user_id.to_numpy();r=order.model_rank.to_numpy();labels=order.label.to_numpy()
    starts=np.r_[0,np.flatnonzero(u[1:]!=u[:-1])+1]
    positives=np.array([POS[s][int(z)] for z in u[starts]],dtype=np.int32)
    assert len(starts)==len(POS[s]) and np.all(positives>0)
    results=[]
    for k in KS:
        mask=r<=k
        hits=int(labels[mask].sum())
        userhits=np.add.reduceat((labels*mask).astype(np.int32),starts)
        dcg=np.add.reduceat(np.where(mask,labels/np.log2(r+1),0.),starts)
        discounts=np.r_[0.,np.cumsum(1/np.log2(np.arange(2,k+2)))]
        idcg=discounts[np.minimum(positives,k)]
        results.append(dict(split=s,method=method,k=k,hits=hits,eligible_positive_pairs=ELIGIBLE[s],
            recall_micro=hits/ELIGIBLE[s],hit_rate=float(np.mean(userhits>0)),ndcg=float(np.mean(dcg/idcg)),user_coverage=1.0))
    ap=float(average_precision_score(Y[s],scores))
    return results,ap,order

# Check that Notebook 10's baseline scores reproduce its saved metrics.
old=REPORT/'notebook10_ranking_metrics.csv'
if old.exists():
    reference=pd.read_csv(old)
    for method in BASE_METHODS:
        rows,_,_=evaluate('validation',method,BASE['validation'][method])
        for row in rows:
            match=reference.query('split=="validation" and method==@method and k==@row["k"]') if False else reference[(reference.split=='validation')&(reference.method==method)&(reference.k==row['k'])]
            assert len(match)==1 and int(match.iloc[0].hits)==row['hits'],(method,row['k'],'baseline mismatch')
    print('Notebook 10 baseline metric reproducibility: PASS')

Notebook 10 baseline metric reproducibility: PASS


## 4. PyTorch DeepFM

Numerical features are represented as scalar fields with learned linear terms and FM latent vectors. Three category IDs use embeddings. Deep branch combines embedded categorical vectors with normalized numerical inputs. Train with sampled negatives (retain **all positives**) and `BCEWithLogitsLoss`; evaluation always uses **all 500 candidates per user**. Sampling is training-only.

In [5]:
class DeepFM(nn.Module):
    def __init__(self,n_num,cards,dim=8,hidden=64,dropout=.35):
        super().__init__();self.dim=dim;self.n_num=n_num
        self.num_linear=nn.Parameter(torch.zeros(n_num));self.num_fm=nn.Parameter(torch.randn(n_num,dim)*.01)
        self.cat_linear=nn.ModuleList([nn.Embedding(c,1,padding_idx=0) for c in cards])
        self.cat_fm=nn.ModuleList([nn.Embedding(c,dim,padding_idx=0) for c in cards])
        self.bias=nn.Parameter(torch.zeros(()))
        self.deep=nn.Sequential(nn.Linear(n_num+len(cards)*dim,hidden),nn.ReLU(),nn.Dropout(dropout),
                                nn.Linear(hidden,hidden//2),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden//2,1))
        for e in list(self.cat_linear)+list(self.cat_fm):nn.init.normal_(e.weight,std=.01);e.weight.data[0].zero_()
    def forward(self,x,cat):
        linear=(x*self.num_linear).sum(1)+self.bias
        embeddings=[x.unsqueeze(-1)*self.num_fm.unsqueeze(0)]
        deep_c=[]
        for j,(e1,e2) in enumerate(zip(self.cat_linear,self.cat_fm)):
            linear=linear+e1(cat[:,j]).squeeze(-1)
            z=e2(cat[:,j]);embeddings.append(z.unsqueeze(1));deep_c.append(z)
        z=torch.cat(embeddings,dim=1)
        fm=.5*((z.sum(dim=1)**2-z.square().sum(dim=1)).sum(dim=1))
        deep=self.deep(torch.cat([x]+deep_c,dim=1)).squeeze(-1)
        return linear+fm+deep

EMBED_DIM=8; HIDDEN=64; DROPOUT=.35
BATCH_SIZE=8192 if DEVICE.type=='cuda' else 4096
EVAL_BATCH_SIZE=16384 if DEVICE.type=='cuda' else 8192
MAX_EPOCHS=12;PATIENCE=3;NEG_PER_POS=60;MIN_NEG_PER_USER=20
LR=.001;WEIGHT_DECAY=1e-4

def train_indices():
    d=DATA['train'];rng=np.random.default_rng(SEED)
    indices=[]
    for _,sub in d.groupby('user_id',sort=False).indices.items():
        idx=np.asarray(sub);pos=idx[Y['train'][idx]>0];neg=idx[Y['train'][idx]==0]
        n=min(len(neg),max(MIN_NEG_PER_USER,len(pos)*NEG_PER_POS))
        chosen=rng.choice(neg,size=n,replace=False) if n else np.array([],dtype=int)
        indices.extend(pos.tolist());indices.extend(chosen.tolist())
    indices=np.asarray(indices,dtype=np.int64);rng.shuffle(indices)
    return indices
TRAIN_IDX=train_indices()
print('Sampled training rows:',len(TRAIN_IDX),'positives:',int(Y['train'][TRAIN_IDX].sum()),
      'sampled positive rate:',round(float(Y['train'][TRAIN_IDX].mean()),5))

def predict(model,s,ablate_categories=False):
    model.eval();out=np.empty(len(Y[s]),dtype=np.float32)
    with torch.no_grad():
        for st in range(0,len(out),EVAL_BATCH_SIZE):
            en=min(st+EVAL_BATCH_SIZE,len(out))
            a=torch.from_numpy(XNUM[s][st:en]).to(DEVICE)
            c=torch.from_numpy(XCAT[s][st:en]).to(DEVICE)
            if ablate_categories:c=torch.zeros_like(c)
            out[st:en]=model(a,c).detach().cpu().numpy()
    return out

def fit_model(use_category=True,epochs=MAX_EPOCHS):
    model=DeepFM(len(NUMS),CARD,dim=EMBED_DIM,hidden=HIDDEN,dropout=DROPOUT).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    # Sampled negatives reduce imbalance without using positive class weights (avoid double correction).
    loader=DataLoader(TensorDataset(torch.from_numpy(XNUM['train'][TRAIN_IDX]),
                        torch.from_numpy(XCAT['train'][TRAIN_IDX]),torch.from_numpy(Y['train'][TRAIN_IDX])),
                        batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS)
    best=-np.inf;best_state=None;best_epoch=0;wait=0;history=[]
    for epoch in range(1,epochs+1):
        model.train();losses=[]
        for xn,xc,yy in loader:
            xn=xn.to(DEVICE);xc=xc.to(DEVICE);yy=yy.to(DEVICE)
            if not use_category:xc=torch.zeros_like(xc)
            opt.zero_grad(set_to_none=True)
            logits=model(xn,xc)
            loss=nn.functional.binary_cross_entropy_with_logits(logits,yy)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.)
            opt.step();losses.append(float(loss.item()))
        scores=predict(model,'validation',ablate_categories=not use_category)
        rows,ap,_=evaluate('validation','deepfm',scores)
        ndcg20=next(r['ndcg'] for r in rows if r['k']==20)
        history.append(dict(epoch=epoch,train_sampled_bce=float(np.mean(losses)),validation_ndcg20=ndcg20,validation_pr_auc=ap))
        print(f'epoch={epoch:02d} train_BCE={np.mean(losses):.5f} val_NDCG@20={ndcg20:.5f} val_AP={ap:.5f}')
        if ndcg20>best+1e-6:
            best=ndcg20;best_epoch=epoch;best_state=copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()});wait=0
        else:
            wait+=1
            if wait>=PATIENCE:break
    model.load_state_dict(best_state)
    return model,pd.DataFrame(history),best_epoch

t0=time.perf_counter()
model,history,best_epoch=fit_model()
train_seconds=time.perf_counter()-t0
print('DeepFM best epoch:',best_epoch,'seconds:',round(train_seconds,1))
display(history)

Sampled training rows: 180836 positives: 2131 sampled positive rate: 0.01178
epoch=01 train_BCE=0.44691 val_NDCG@20=0.00637 val_AP=0.00087
epoch=02 train_BCE=0.17354 val_NDCG@20=0.00954 val_AP=0.00137
epoch=03 train_BCE=0.07700 val_NDCG@20=0.03403 val_AP=0.00819
epoch=04 train_BCE=0.06248 val_NDCG@20=0.04623 val_AP=0.01237
epoch=05 train_BCE=0.06102 val_NDCG@20=0.05020 val_AP=0.01353
epoch=06 train_BCE=0.05749 val_NDCG@20=0.05232 val_AP=0.01417
epoch=07 train_BCE=0.05577 val_NDCG@20=0.05312 val_AP=0.01511
epoch=08 train_BCE=0.05496 val_NDCG@20=0.05352 val_AP=0.01562
epoch=09 train_BCE=0.05461 val_NDCG@20=0.05346 val_AP=0.01583
epoch=10 train_BCE=0.05412 val_NDCG@20=0.05402 val_AP=0.01530
epoch=11 train_BCE=0.05265 val_NDCG@20=0.05431 val_AP=0.01596
epoch=12 train_BCE=0.05341 val_NDCG@20=0.05432 val_AP=0.01495
DeepFM best epoch: 12 seconds: 71.1


,epoch,train_sampled_bce,validation_ndcg20,validation_pr_auc
0,1,0.446906,0.006372,0.000868
1,2,0.173535,0.009540,0.001374
2,3,0.076999,0.034026,0.008193
3,4,0.062480,0.046228,0.012374
4,5,0.061019,0.050195,0.013534
5,6,0.057493,0.052315,0.014170
6,7,0.055766,0.053116,0.015107
7,8,0.054964,0.053518,0.015616
8,9,0.054608,0.053461,0.015825
9,10,0.054116,0.054016,0.015299


## 5. Validation comparison and score fusion

Fusion combines **within-user percentile ranks**, not uncalibrated raw LightGBM probabilities and DeepFM logits. Choose fusion weight using Validation NDCG@20 only. No test inspection until selection is frozen.

In [6]:
def user_percentile(s,scores):
    d=DATA[s]
    # Stable percentile in [0,1], per user; every user has exactly 500 candidates.
    t=pd.DataFrame({'user_id':d.user_id.to_numpy(),'score':np.asarray(scores),
                    'original_rank':d.retrieval_rank.to_numpy(),'item_id':d.item_id.to_numpy()})
    t['p']=t.groupby('user_id',sort=False)['score'].rank(method='first',pct=True)
    return t.p.to_numpy(dtype=np.float32)

val_deep=predict(model,'validation')
val_binary=BASE['validation']['binary']
PRED={('validation','deepfm'):val_deep}
rows,ap,_=evaluate('validation','deepfm',val_deep)
val_metrics=list(rows);val_ap=[dict(split='validation',method='deepfm',pr_auc=ap)]
val_b_pct=user_percentile('validation',val_binary)
val_d_pct=user_percentile('validation',val_deep)
weights=[0.,.25,.5,.75,1.]
fusion_trials=[]
for alpha in weights:
    fused=alpha*val_d_pct+(1-alpha)*val_b_pct
    m,a,_=evaluate('validation','fusion',fused)
    fusion_trials.append(dict(deepfm_weight=alpha,validation_ndcg20=next(r['ndcg'] for r in m if r['k']==20),
                              validation_recall20=next(r['recall_micro'] for r in m if r['k']==20),validation_pr_auc=a))
trial_df=pd.DataFrame(fusion_trials)
# Ties choose smaller DeepFM weight, preserving proven LightGBM baseline.
chosen=trial_df.sort_values(['validation_ndcg20','deepfm_weight'],ascending=[False,True],kind='stable').iloc[0]
ALPHA=float(chosen.deepfm_weight)
val_fused=ALPHA*val_d_pct+(1-ALPHA)*val_b_pct
PRED[('validation','fusion')]=val_fused
m,a,_=evaluate('validation','fusion',val_fused)
val_metrics.extend(m);val_ap.append(dict(split='validation',method='fusion',pr_auc=a))
print('Selected DeepFM fusion weight:',ALPHA)
display(trial_df)
for method in BASE_METHODS:
    m,a,_=evaluate('validation',method,BASE['validation'][method]);val_metrics.extend(m);val_ap.append(dict(split='validation',method=method,pr_auc=a))
display(pd.DataFrame(val_metrics).pivot(index='method',columns='k',values=['recall_micro','ndcg','hit_rate']).round(5))

Selected DeepFM fusion weight: 1.0


,deepfm_weight,validation_ndcg20,validation_recall20,validation_pr_auc
0,0.00,0.050396,0.070399,0.007721
1,0.25,0.052246,0.074390,0.009313
2,0.50,0.054047,0.077938,0.009497
3,0.75,0.053997,0.077605,0.009548
4,1.00,0.054324,0.077827,0.008407


recall_micro                       ndcg                    \
k                     20       50       100      20       50       100   
method                                                                   
binary            0.07106  0.10920  0.14246  0.05085  0.06153  0.06937   
deepfm            0.07783  0.11386  0.14490  0.05432  0.06388  0.07032   
fusion            0.07783  0.11386  0.14490  0.05432  0.06388  0.07032   
lambdarank        0.06763  0.10399  0.13625  0.04886  0.05912  0.06603   
rrf_baseline      0.06818  0.10277  0.13137  0.04528  0.05455  0.06097   

             hit_rate                    
k                 20       50       100  
method                                   
binary        0.13071  0.18430  0.23052  
deepfm        0.14520  0.19278  0.22970  
fusion        0.14520  0.19278  0.22970  
lambdarank    0.12961  0.18321  0.22313  
rrf_baseline  0.13126  0.17801  0.21849

## 6. Feature ablation (validation-only)

A fast **inference-time category masking** diagnostic measures sensitivity of the trained DeepFM to category embeddings. This is *not* a controlled retraining ablation: masking can introduce distribution shift. Optionally enable `RUN_RETRAIN_ABLATION=True` to retrain without category embeddings and compare fairly.

In [7]:
masked=predict(model,'validation',ablate_categories=True)
masked_rows,masked_ap,_=evaluate('validation','deepfm_category_masked',masked)
ABLATION=[dict(variant='full_deepfm',ndcg20=next(x['ndcg'] for x in rows if x['k']==20),pr_auc=ap),
          dict(variant='category_masked_inference_only',ndcg20=next(x['ndcg'] for x in masked_rows if x['k']==20),pr_auc=masked_ap)]
RUN_RETRAIN_ABLATION=False
if RUN_RETRAIN_ABLATION:
    no_cat_model,no_cat_history,no_cat_epoch=fit_model(use_category=False)
    no_cat_scores=predict(no_cat_model,'validation',ablate_categories=True)
    ab_rows,ab_ap,_=evaluate('validation','deepfm_no_category_retrained',no_cat_scores)
    ABLATION.append(dict(variant='no_category_retrained',ndcg20=next(x['ndcg'] for x in ab_rows if x['k']==20),pr_auc=ab_ap))
    del no_cat_model
display(pd.DataFrame(ABLATION))

,variant,ndcg20,pr_auc
0,full_deepfm,0.054324,0.014950
1,category_masked_inference_only,0.053072,0.013774


## 7. Freeze selection, then evaluate Test once

No test data is used for fitting, early stopping, choosing DeepFM hyperparameters, fusion weight or ablation decisions.

In [8]:
# Optional Train metrics are in-sample and should not be used for model selection.
ALL_METRICS=list(val_metrics);ALL_AP=list(val_ap)
for s in ('train','test'):
    deep=predict(model,s)
    fused=ALPHA*user_percentile(s,deep)+(1-ALPHA)*user_percentile(s,BASE[s]['binary'])
    PRED[(s,'deepfm')]=deep;PRED[(s,'fusion')]=fused
    for method in ('rrf_baseline','binary','lambdarank','deepfm','fusion'):
        scores=BASE[s][method] if method in BASE_METHODS else PRED[(s,method)]
        m,a,_=evaluate(s,method,scores)
        ALL_METRICS.extend(m);ALL_AP.append(dict(split=s,method=method,pr_auc=a))
metrics_df=pd.DataFrame(ALL_METRICS);ap_df=pd.DataFrame(ALL_AP)
print('Test ranking metrics (confirmation):')
display(metrics_df.query('split=="test"').pivot(index='method',columns='k',values=['recall_micro','ndcg','hit_rate']).round(5))
display(ap_df.query('split=="test"'))

Test ranking metrics (confirmation):


recall_micro                       ndcg                    \
k                     20       50       100      20       50       100   
method                                                                   
binary            0.07256  0.10713  0.14244  0.05400  0.06334  0.07109   
deepfm            0.07487  0.10969  0.14524  0.05595  0.06525  0.07272   
fusion            0.07487  0.10969  0.14524  0.05595  0.06525  0.07272   
lambdarank        0.06489  0.10019  0.13550  0.04683  0.05607  0.06380   
rrf_baseline      0.06672  0.09910  0.12978  0.04718  0.05590  0.06252   

             hit_rate                    
k                 20       50       100  
method                                   
binary        0.13158  0.17945  0.22676  
deepfm        0.13466  0.18225  0.22872  
fusion        0.13466  0.18225  0.22872  
lambdarank    0.12010  0.16993  0.21809  
rrf_baseline  0.12374  0.17105  0.21221

,split,method,pr_auc
10,test,rrf_baseline,0.006218
11,test,binary,0.014597
12,test,lambdarank,0.009456
13,test,deepfm,0.015456
14,test,fusion,0.008202


## 8. Export Parquet predictions, checkpoint, reports

Prediction Parquet includes `(user_id,item_id,label,score,original_rank,model_rank)` for DeepFM and Fusion; CSV metrics and JSON report include configuration, limitations and audit.

In [9]:
ckpt=MODEL_DIR/'notebook11_deepfm_checkpoint.pt'
torch.save({'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
    'num_features':NUMS,'categorical_features':CATS,'category_vocabularies':VOCABS,
    'category_cardinalities':CARD,'numeric_median':med.tolist(),'numeric_log_mask':logmask.tolist(),
    'numeric_clip_low':lo.tolist(),'numeric_clip_high':hi.tolist(),
    'numeric_mean':mu.tolist(),'numeric_std':sigma.tolist(),
    'embedding_dim':EMBED_DIM,'hidden_dim':HIDDEN,'dropout':DROPOUT,
    'best_epoch':best_epoch,'fusion_deepfm_weight':ALPHA,'seed':SEED},ckpt)
exports=[]
for s in SPLITS:
    for method in ('deepfm','fusion'):
        _,_,pred=evaluate(s,method,PRED[(s,method)])
        p=OUT/f'{s}_deepfm_{method}_top500.parquet'
        pred.to_parquet(p,index=False,compression='zstd')
        exports.append(dict(split=s,method=method,path=p.name,rows=len(pred),size_mb=round(p.stat().st_size/1048576,2)))
metrics_df.to_csv(REPORT/'notebook11_ranking_metrics.csv',index=False)
ap_df.to_csv(REPORT/'notebook11_pr_auc.csv',index=False)
trial_df.to_csv(REPORT/'notebook11_fusion_search.csv',index=False)
pd.DataFrame(ABLATION).to_csv(REPORT/'notebook11_feature_ablation.csv',index=False)
history.to_csv(REPORT/'notebook11_training_history.csv',index=False)
pd.DataFrame(audit).to_csv(REPORT/'notebook11_data_audit.csv',index=False)
report={'notebook':'11_deepfm_fine_ranking','seed':SEED,'device':str(DEVICE),
 'features':{'numerical':NUMS,'categorical':CATS,'embedding_dim':EMBED_DIM,'hidden_dim':HIDDEN},
 'training':{'epochs_requested':MAX_EPOCHS,'best_epoch':best_epoch,'patience':PATIENCE,
             'negative_per_positive':NEG_PER_POS,'min_negative_per_user':MIN_NEG_PER_USER,
             'sampled_training_rows':len(TRAIN_IDX),'seconds':round(train_seconds,2)},
 'model_selection':{'metric':'validation full-eligible-user NDCG@20','fusion_weight_deepfm':ALPHA,
                    'test_used_for_selection':False},
 'data_audit':audit,'ranking_metrics':metrics_df.to_dict('records'),
 'pr_auc':ap_df.to_dict('records'),'fusion_trials':fusion_trials,'feature_ablation':ABLATION,
 'training_history':history.to_dict('records'),'prediction_exports':exports,
 'checkpoint':ckpt.name,
 'limitations':['Weak implicit negatives; no impression logs or calibrated CTR',
  'Train-only sampled negatives; evaluation always on all 500 candidates per user',
  'Candidate ranking cannot recover positives missing from retrieval',
  'Split-level PIT inherited from Notebook 09, not event-level PIT',
  'Inference-time category masking is not equivalent to retraining ablation',
  'No online latency, causal CTR/CVR, or production performance claim']}
p=REPORT/'notebook11_deepfm_report.json'
p.write_text(json.dumps(report,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('Saved:',p,'Checkpoint:',ckpt)
display(pd.DataFrame(exports))

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook11_deepfm_report.json Checkpoint: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\models\notebook11_deepfm_checkpoint.pt


,split,method,path,rows,size_mb
0,train,deepfm,train_deepfm_deepfm_top500.parquet,2055000,13.38
1,train,fusion,train_deepfm_fusion_top500.parquet,2055000,5.93
2,validation,deepfm,validation_deepfm_deepfm_top500.parquet,1828500,12.10
3,validation,fusion,validation_deepfm_fusion_top500.parquet,1828500,5.37
4,test,deepfm,test_deepfm_deepfm_top500.parquet,1786000,11.73
5,test,fusion,test_deepfm_fusion_top500.parquet,1786000,5.20


## 9. MVP completion checklist

- [ ] Train-only preprocessing and vocabulary; validation early stopping
- [ ] Baselines reproduce Notebook 10 metrics
- [ ] All candidate users/groups remain at 500; full eligible-positive metric denominators
- [ ] Test inspected only after DeepFM and fusion selection
- [ ] Report / prediction Parquets / checkpoint saved

**Next:** upload `data/reports/notebook11_deepfm_report.json`. If DeepFM does not beat LightGBM, keep LightGBM Binary as the production-style MVP winner; do not force a deep model into the stack.